<a href="https://colab.research.google.com/github/ahmadfa100/-Ahmad-bani-Hamad/blob/main/fine_tuning_examplestest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    pipeline,
    set_seed
)
from datasets import load_dataset, Dataset
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

print("Starting updated fine-tuning examples...")

Starting updated fine-tuning examples...


In [ ]:
def compute_metrics(eval_pred):
    """Calculate evaluation metrics"""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')
    acc = accuracy_score(labels, predictions)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [ ]:
print("\n--- Example 1: Fine-tuning for Text Classification (Sentiment Analysis) ---")
print("Loading dataset...")

try:
    dataset = load_dataset("imdb")
    train_dataset = dataset["train"].shuffle(seed=42).select(range(1000))
    eval_dataset = dataset["test"].shuffle(seed=42).select(range(200))
    print("IMDB dataset loaded successfully")
except Exception as e:
    print(f"Error loading dataset: {e}")
    print("Using synthetic dataset...")

    synthetic_data = {
        'text': [
            "This movie is amazing! I loved every minute of it.",
            "Terrible film, waste of time.",
            "Great acting and wonderful story.",
            "Boring and predictable plot.",
            "Excellent cinematography and direction.",
            "Poor script and bad acting.",
            "One of the best movies I've ever seen.",
            "Disappointing and overrated.",
            "Fantastic performances by all actors.",
            "Not worth watching, very dull."
        ],
        'label': [1, 0, 1, 0, 1, 0, 1, 0, 1, 0]
    }

    train_dataset = Dataset.from_dict(synthetic_data)
    eval_dataset = Dataset.from_dict({
        'text': synthetic_data['text'][:4],
        'label': synthetic_data['label'][:4]
    })


--- Example 1: Fine-tuning for Text Classification (Sentiment Analysis) ---
Loading dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

IMDB dataset loaded successfully


In [ ]:
print("Loading tokenizer and model...")
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_eval_dataset = eval_dataset.map(tokenize_function, batched=True)

tokenized_train_dataset = tokenized_train_dataset.rename_column("label", "labels")
tokenized_eval_dataset = tokenized_eval_dataset.rename_column("label", "labels")
tokenized_train_dataset = tokenized_train_dataset.remove_columns(["text"])
tokenized_eval_dataset = tokenized_eval_dataset.remove_columns(["text"])
tokenized_train_dataset.set_format("torch")
tokenized_eval_dataset.set_format("torch")

Loading tokenizer and model...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [ ]:
import os
from transformers import TrainingArguments, Trainer

os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs_sentiment"

training_args = TrainingArguments(
    output_dir="./results_sentiment",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print("Training Sentiment Analysis model...")
trainer.train()
print("Training completed!")
print("Evaluation:", trainer.evaluate())

Training Sentiment Analysis model...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.561685,0.801571,0.725000,0.718383,0.759752,0.725000
2,0.250170,0.745060,0.780000,0.779538,0.786788,0.780000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Training completed!


Evaluation: {'eval_loss': 0.7450602650642395, 'eval_accuracy': 0.78, 'eval_f1': 0.7795375838254429, 'eval_precision': 0.7867878787878788, 'eval_recall': 0.78, 'eval_runtime': 0.9127, 'eval_samples_per_second': 219.137, 'eval_steps_per_second': 54.784, 'epoch': 2.0}


In [ ]:
import os
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    pipeline,
)
from datasets import Dataset

print("\n--- Example 2: Text Generation ---")

model_name_gen = "distilgpt2"
tokenizer_gen = AutoTokenizer.from_pretrained(model_name_gen)
model_gen = AutoModelForCausalLM.from_pretrained(model_name_gen)

if tokenizer_gen.pad_token is None:
    tokenizer_gen.pad_token = tokenizer_gen.eos_token

text_data = [
    "The quick brown fox jumps over the lazy dog.",
    "Fine-tuning allows adapting pre-trained models.",
    "Generative models create new and creative content.",
    "AI is transforming many industries."
]

text_dataset = Dataset.from_dict({"text": text_data})

def tokenize_text_data(examples):
    return tokenizer_gen(examples["text"], truncation=True, padding=True, max_length=128)

tokenized_text_dataset = text_dataset.map(
    tokenize_text_data,
    batched=True,
    remove_columns=["text"]
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer_gen, mlm=False)

os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs_text_gen"

training_args_gen = TrainingArguments(
    output_dir="./results_text_gen",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    save_strategy="epoch",
    logging_steps=5,
    learning_rate=5e-5,
    warmup_steps=50,
    report_to="none",
)

trainer_gen = Trainer(
    model=model_gen,
    args=training_args_gen,
    data_collator=data_collator,
    train_dataset=tokenized_text_dataset,
    processing_class=tokenizer_gen,
)

trainer_gen.train()

generator = pipeline(
    "text-generation",
    model=model_gen,
    tokenizer=tokenizer_gen,
    device=0 if torch.cuda.is_available() else -1
)

output = generator(
    "The model will",
    max_length=50,
    num_return_sequences=1,
    temperature=0.7,
    do_sample=True
)
print(output[0]["generated_text"])


--- Example 2: Text Generation ---


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Map:   0%|          | 0/4 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_length', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The model will be released in the US on January 20th, 2016.


The new model will be produced at the US and UK and will include a number of different configurations of the models, including the new model. After the first release of the new model, the model will be made available in the US and UK, as well as in the UK, in the UK and Europe.
The model will have a full set of colors, with different colors, and they will be made available in the US and UK.
The new model will be made available in the US and UK, and will include an array of colors, including a number of colors, including a number of colors, including a number of colors, including a number of colors, including a number of colors, including a number of colors, including a number of colors, including a number of colors, including a number of colors, including a number of colors, including a number of colors, including a number of colors, including a number of colors, including a number of colors, including a number of colors, in

In [ ]:
print("\n--- Example 3: LoRA Fine-tuning ---")

try:
    from peft import LoraConfig, get_peft_model, TaskType

    base_model = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased",
        num_labels=2
    )

    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        inference_mode=False,
        r=8,
        lora_alpha=32,
        lora_dropout=0.1,
        target_modules=["q_lin", "k_lin", "v_lin"],
    )

    lora_model = get_peft_model(base_model, lora_config)

    print("LoRA model created!")
    print(f"Trainable parameters: {lora_model.get_nb_trainable_parameters()}")
except ImportError:
    print("Install PEFT with: pip install peft")


print("Setting up Trainer for LoRA model...")

training_args_lora = TrainingArguments(
    output_dir="./results_lora",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    report_to="none",
)

trainer_lora = Trainer(
    model=lora_model,
    args=training_args_lora,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print("Training LoRA model...")
trainer_lora.train()

print("LoRA Training completed!")
print("LoRA Evaluation:", trainer_lora.evaluate())


--- Example 3: LoRA Fine-tuning ---


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


LoRA model created!
Trainable parameters: (813314, 67768324)
Setting up Trainer for LoRA model...
Training LoRA model...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.589076,0.516809,0.765000,0.756702,0.794842,0.765000
2,0.400871,0.412790,0.820000,0.820054,0.820272,0.820000


LoRA Training completed!


LoRA Evaluation: {'eval_loss': 0.4127902090549469, 'eval_accuracy': 0.82, 'eval_f1': 0.8200540486437794, 'eval_precision': 0.8202721088435374, 'eval_recall': 0.82, 'eval_runtime': 1.7339, 'eval_samples_per_second': 115.35, 'eval_steps_per_second': 28.838, 'epoch': 2.0}


In [ ]:
print("\nNotes:")
print("1. Install dependencies: pip install transformers datasets torch scikit-learn")
print("2. For LoRA: pip install peft")
print("3. GPU is recommended for large models")
print("4. Adjust hyperparameters according to your resources")


Notes:
1. Install dependencies: pip install transformers datasets torch scikit-learn
2. For LoRA: pip install peft
3. GPU is recommended for large models
4. Adjust hyperparameters according to your resources
